# Bypass Pipeline Capacity Dashboard

This notebook calculates the amount of crude oil that can bypass the Strait of Hormuz using alternative pipeline routes.

## 1. Load Libraries

In [22]:
import sys
sys.path.insert(0, '/Users/astonchia/Documents/GitHub/AI-Immigration')
sys.path.insert(0, '/Users/astonchia/Documents/GitHub/AI-Immigration/Data_loaders')

In [23]:
import pandas as pd
import plotly.express as px

In [24]:
from Data_loaders.dataloader_pipeline import load_pipeline_data

df = load_pipeline_data()
df.head()


[DATA NOTICE] Pipeline capacity data: using representative sample datastraitofhormuz.report requires account login for CSV. Figures below are from EIA published capacity tables + Kpler/IEA utilisation estimates. Replace with CSV once downloaded from the site.
 I need to replace with real CVS data



/var/folders/62/ldhc8pgx7t33nbrhmd2gx2ww0000gn/T/ipykernel_64333/2861605737.py:3: UserWarning: 
[DATA NOTICE] Pipeline capacity data: using representative sample datastraitofhormuz.report requires account login for CSV. Figures below are from EIA published capacity tables + Kpler/IEA utilisation estimates. Replace with CSV once downloaded from the site.
 I need to replace with real CVS data

  df = load_pipeline_data()


,date,pipeline,max_capacity_mbpd,utilisation_pct,flow_mbpd
0,2019-01-01,Saudi East-West (Petroline),5.0,68.6,3.43
1,2019-02-01,Saudi East-West (Petroline),5.0,66.0,3.30
2,2019-03-01,Saudi East-West (Petroline),5.0,65.5,3.27
3,2019-04-01,Saudi East-West (Petroline),5.0,61.7,3.09
4,2019-05-01,Saudi East-West (Petroline),5.0,65.0,3.25


## 2. Load Pipeline Data

In [25]:
from Data_loaders.dataloader_pipeline import load_pipeline_data

df = load_pipeline_data()

df.head()


[DATA NOTICE] Pipeline capacity data: using representative sample datastraitofhormuz.report requires account login for CSV. Figures below are from EIA published capacity tables + Kpler/IEA utilisation estimates. Replace with CSV once downloaded from the site.
 I need to replace with real CVS data



/var/folders/62/ldhc8pgx7t33nbrhmd2gx2ww0000gn/T/ipykernel_64333/332662574.py:3: UserWarning: 
[DATA NOTICE] Pipeline capacity data: using representative sample datastraitofhormuz.report requires account login for CSV. Figures below are from EIA published capacity tables + Kpler/IEA utilisation estimates. Replace with CSV once downloaded from the site.
 I need to replace with real CVS data

  df = load_pipeline_data()


,date,pipeline,max_capacity_mbpd,utilisation_pct,flow_mbpd
0,2019-01-01,Saudi East-West (Petroline),5.0,68.6,3.43
1,2019-02-01,Saudi East-West (Petroline),5.0,66.0,3.30
2,2019-03-01,Saudi East-West (Petroline),5.0,65.5,3.27
3,2019-04-01,Saudi East-West (Petroline),5.0,61.7,3.09
4,2019-05-01,Saudi East-West (Petroline),5.0,65.0,3.25


## 3. Calculate Available Bypass Capacity

In [26]:
print(df.columns.tolist())

['date', 'pipeline', 'max_capacity_mbpd', 'utilisation_pct', 'flow_mbpd']


In [27]:
NORMAL_SOH_FLOW = 20

total_bypass = df["max_capacity_mbpd"].sum()

gap = NORMAL_SOH_FLOW - total_bypass

coverage = total_bypass / NORMAL_SOH_FLOW * 100

print("Coverage:", coverage)

Coverage: 3401.9999999999995


## 4. Visualise Pipeline Capacity

In [30]:
import plotly.graph_objects as go

NORMAL_SOH_FLOW = 20

# Get average capacity per pipeline
latest = df.groupby('pipeline')['max_capacity_mbpd'].mean().reset_index()
latest = latest.sort_values('max_capacity_mbpd', ascending=True)
total_bypass = latest['max_capacity_mbpd'].sum()
gap = NORMAL_SOH_FLOW - total_bypass

fig = go.Figure()

# Individual pipeline bars
colors = ['#2A5F9E', '#357ABD', '#4A90E2']
for i, row in latest.iterrows():
    fig.add_trace(go.Bar(
        x=[row['max_capacity_mbpd']],
        y=[row['pipeline']],
        orientation='h',
        marker_color=colors[i % len(colors)],
        showlegend=False,
        text=f"~{row['max_capacity_mbpd']:.1f}M bbl/day",
        textposition='outside',
        textfont=dict(color='white', size=12)
    ))

# Total bypass filled bar
fig.add_trace(go.Bar(
    x=[total_bypass],
    y=['<b>Total Bypass</b>'],
    orientation='h',
    marker_color='#4A90E2',
    showlegend=False,
    text=f'~{total_bypass:.0f}M bbl/day',
    textposition='inside',
    textfont=dict(color='white', size=13)
))

# Gap bar (stacked onto total)
fig.add_trace(go.Bar(
    x=[gap],
    y=['<b>Total Bypass</b>'],
    orientation='h',
    marker_color='#2d3748',
    showlegend=False,
    text=f'Gap: ~{gap:.0f}M bbl/day',
    textposition='inside',
    textfont=dict(color='red', size=13)
))

fig.update_layout(
    title=dict(text='Bypass Pipeline Capacity', font=dict(size=20, color='white')),
    barmode='stack',
    plot_bgcolor='#1a202c',
    paper_bgcolor='#1a202c',
    font=dict(color='white'),
    xaxis=dict(
        range=[0, 22],
        showgrid=False,
        zeroline=False,
        tickfont=dict(color='white')
    ),
    yaxis=dict(showgrid=False),
    height=450,
    margin=dict(l=250, r=150, t=80, b=80),
    annotations=[
        dict(
            x=20, y=1.08,
            text='Normal strait flow: ~20M bbl/day',
            showarrow=False,
            font=dict(color='white', size=11),
            xanchor='right'
        ),
        dict(
            x=0, y=-0.22,
            text=f'⚠ ~{gap:.0f}M bbl/day cannot be rerouted via pipelines',
            showarrow=False,
            font=dict(color='red', size=12),
            xanchor='left',
            xref='paper', yref='paper'
        )
    ]
)

fig.show()